# SWE-GraphBench: reproduce the audit and benchmark

Companion notebook for the Paper Track writeup *SWE-GraphBench: Auditing and Rebuilding Code Graphs for Local Software-Engineering Agents*.

Inputs:
* the competition data (`/kaggle/input/gemma-4-developer-agent`): `tasks.jsonl`, `graphs/`, `embeddings/`
* the SWE-GraphBench dataset (`nateshingleton/swe-graphbench`): package, scripts, gold symbols, 127 rebuilt graphs

Everything below runs on CPU. Steps that need the upstream git repositories (gold construction, rebuilding graphs) are shipped precomputed and demonstrated on one commit at the end.

In [ ]:
import os, shutil, subprocess, sys, json
from pathlib import Path
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'rank_bm25', 'unidiff'], check=True)
COMP = Path('/kaggle/input/gemma-4-developer-agent')

def find_dir(name):
    # locate a folder of the SWE-GraphBench dataset regardless of how Kaggle laid out the unzipped upload
    hits = [p for p in Path('/kaggle/input').rglob(name) if p.is_dir() and 'gemma-4-developer-agent' not in str(p)]
    assert hits, f'{name} not found under /kaggle/input'
    return sorted(hits, key=lambda p: len(str(p)))[0]

ROOT = Path('/kaggle/working/swegb'); ROOT.mkdir(exist_ok=True)
for d in ['swegraphbench', 'scripts', 'tests']:
    if (ROOT / d).exists(): shutil.rmtree(ROOT / d)
    shutil.copytree(find_dir(d), ROOT / d)
(ROOT / 'results').mkdir(exist_ok=True)
gold = [p for p in Path('/kaggle/input').rglob('gold.jsonl') if 'gemma-4-developer-agent' not in str(p)][0]
shutil.copy(gold, ROOT / 'results' / 'gold.jsonl')
if not (ROOT / 'results' / 'rebuilt_graphs').exists():
    os.symlink(find_dir('rebuilt_graphs'), ROOT / 'results' / 'rebuilt_graphs')
if not (ROOT / 'data').exists():
    os.symlink(COMP, ROOT / 'data')
os.chdir(ROOT); print(sorted(os.listdir(ROOT)))
print('rebuilt graphs:', len(list((ROOT / 'results' / 'rebuilt_graphs').glob('*.json'))))

## 1. Builder unit test

In [ ]:
!python tests/test_build_graph.py

## 2. Audit of the released graphs and embeddings, and gold coverage

In [ ]:
!python scripts/02_audit_coverage.py | tail -n 60

## 3. Released extractor versus `ast` ground truth (async, hoisting, direction)

In [ ]:
!python scripts/06_node_mapping_audit.py | tail -n 25

## 4. LLM-free localization baselines on the released graphs

In [ ]:
!python scripts/03_baselines_released.py | tail -n 12

## 5. Released versus rebuilt graphs, and the tasks the release left uncovered

In [ ]:
!python scripts/05_compare_graphs.py | tail -n 15
!python scripts/09_paired_stats.py | head -n 20

## 6. Build a graph from source inside this sandbox

The builder has no dependencies beyond the standard library. With internet enabled this cell fetches one task commit of FastAPI and rebuilds its graph; the same call works on any Python repository (for example `/workspace` inside the competition sandbox).

In [ ]:
import time
sha = '016ab760faba1813ba235d5582a014a58bd2f417'
wt = Path('/kaggle/working/fastapi_demo')
try:
    if not wt.exists():
        subprocess.run(['git', 'init', '-q', str(wt)], check=True)
        subprocess.run(['git', '-C', str(wt), 'fetch', '-q', '--depth', '1', 'https://github.com/fastapi/fastapi.git', sha], check=True, timeout=300)
        subprocess.run(['git', '-C', str(wt), 'checkout', '-q', 'FETCH_HEAD'], check=True)
    t0 = time.time()
    out = subprocess.run([sys.executable, '-m', 'swegraphbench.build_graph', str(wt), '/kaggle/working/demo_graph.json', '--name', f'fastapi_{sha}'], capture_output=True, text=True, check=True)
    print(out.stdout); print(f'built in {time.time()-t0:.1f} s')
except Exception as e:
    print('skipped (internet disabled or fetch failed):', e)